# Student Performance

In [26]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [27]:
def extract_summary_features(summary):
    sem_gpas = summary["semester_gpas"]

    # trend features
    avg_gpa = np.mean(sem_gpas)
    gpa_slope = sem_gpas[-1] - sem_gpas[0] if len(sem_gpas) > 1 else 0
    last_gpa = sem_gpas[-1]

    # stability
    volatility = np.std(sem_gpas)

    # credit efficiency
    credit_eff = summary["total_credits_completed"] / max(summary["total_credits_attempted"], 1)

    # domain weakness
    domain = summary["domain_gpas"]
    weakest_domain = min(domain.values())

    return np.array([
        avg_gpa,
        gpa_slope,
        last_gpa,
        volatility,
        credit_eff,
        summary["fail_withdrawal_count"],
        weakest_domain
    ])

In [40]:
model = RandomForestRegressor(
    n_estimators=200,
    max_depth=6
)

model.fit(X_train, y_train)

RandomForestRegressor(max_depth=6, n_estimators=200)

In [29]:
def predict_student(summary_input, student_id, model):
    X = extract_summary_features(summary_input).reshape(1, -1)

    # --- predictions ---
    next_gpa = model.predict(X)[0]

    sem_gpas = summary_input["semester_gpas"]
    avg_trend = sem_gpas[-1] - sem_gpas[0]

    # simulate graduation GPA
    predicted_grad_gpa = (summary_input["cumulative_gpa"] + next_gpa) / 2

    # trajectory
    if avg_trend > 0.2:
        trajectory = "Improving"
    elif avg_trend < -0.2:
        trajectory = "Declining"
    else:
        trajectory = "Stable"

    # risk score (custom formula)
    risk_score = (
        (3.0 - next_gpa) * 0.4 +
        summary_input["fail_withdrawal_count"] * 0.2 +
        max(0, 3.0 - summary_input["domain_gpas"]["MATH"]) * 0.2 +
        volatility_weight(sem_gpas)
    )

    risk_score = max(0, min(1, risk_score))

    if risk_score < 0.3:
        risk_level = "Low"
    elif risk_score < 0.7:
        risk_level = "Medium"
    else:
        risk_level = "High"

    # --- alerts ---
    alerts = []

    if avg_trend < -0.5:
        alerts.append("GPA dropped significantly over semesters")

    if summary_input["fail_withdrawal_count"] >= 2:
        alerts.append("Multiple withdrawals detected — possible overload")

    if summary_input["domain_gpas"]["MATH"] < 3.0:
        alerts.append("Weak performance in MATH domain")

    # --- recommendations ---
    recommendations = []

    if risk_level == "High":
        recommendations.append("Immediate academic advising required")
        recommendations.append("Reduce credit load significantly")

    if summary_input["domain_gpas"]["MATH"] < 3.0:
        recommendations.append("Enroll in MATH tutoring or support sessions")

    if trajectory == "Declining":
        recommendations.append("Focus on workload balance and study planning")

    return {
        "student_id": student_id,
        "predictions": {
            "next_semester_gpa": round(float(next_gpa), 2),
            "predicted_graduation_gpa": round(float(predicted_grad_gpa), 2),
            "trajectory": trajectory,
            "risk_level": risk_level,
            "risk_score": round(float(risk_score), 2)
        },
        "alerts": alerts,
        "recommendations": recommendations
    }

In [30]:
def volatility_weight(sem_gpas):
    return np.std(sem_gpas) * 0.3

In [32]:
with open("input.json") as f:
    data = json.load(f)

result = predict_student(
    data["transcript_summary"],
    data["student_id"],
    model
)

print(result)

{'student_id': 'STU-2021-0042', 'predictions': {'next_semester_gpa': 2.99, 'predicted_graduation_gpa': 3.0, 'trajectory': 'Declining', 'risk_level': 'Medium', 'risk_score': 0.53}, 'alerts': ['GPA dropped significantly over semesters', 'Multiple withdrawals detected — possible overload', 'Weak performance in MATH domain'], 'recommendations': ['Enroll in MATH tutoring or support sessions', 'Focus on workload balance and study planning']}


In [33]:
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print("MAE:", mae)

MAE: 0.25342377319598897


In [34]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("RMSE:", rmse)

RMSE: 0.4891937014768231


In [36]:
import joblib

pipeline = {
    "model": model,
    "feature_function": extract_summary_features
}

joblib.dump(pipeline, "gpa_pipeline.pkl")

['gpa_pipeline.pkl']

In [37]:
pipeline = joblib.load("gpa_pipeline.pkl")

model = pipeline["model"]
extract_summary_features = pipeline["feature_function"]

In [38]:
extract_summary_features

<function __main__.extract_summary_features(summary)>

In [41]:
joblib.dump(model, "student_performance.pkl")

['student_performance.pkl']